<h1 class="alert alert-block alert-info">Build OpenAT GPT2 LLM from scratch.</h1>

<img src="../Images/GPT-Model-Architecture.png" width=“500”>

In [1]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
print("PyTorch version: ", torch.__version__)

PyTorch version:  2.11.0


<div class="alert alert-info">GPT Dataset class.
<span style="color: green"> 

It does the following:

1. It converts the text into tokens.
2. Create Input and Target tensor with size equal to context size.
3. Overlapping is defined using Stride

</span>
</div>

In [2]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, context_size, stride = 0):
        self.input_ids = []
        self.target_ids = []

        # if stride isn't given, then set it equal to the context size
        if stride:
            stride = context_size

        # Get the token ids of input text "txt"
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids)-context_size, stride):
            input_token = token_ids[i: i+context_size]
            target_token = token_ids[i+1: i+1+context_size]

            # Decoding the tokens to see the input and target tokens in English.
            # print(tokenizer.decode(input_token), "-->", tokenizer.decode(target_token))

            self.input_ids.append(torch.tensor(input_token))
            self.target_ids.append(torch.tensor(target_token))
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

<div class="alert alert-block alert-info">Create Tensor Dataloader</div>

In [3]:
def create_datalaoder_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last = True, num_workers=0):
    # initializing the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset 
    dataset = GPTDatasetV1(txt=txt, 
                           tokenizer=tokenizer, 
                           context_size=max_length,
                           stride=stride)
    
    # Create data loader
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

<img src="../Images/atten_with_trainable_weights.png" width=“500”>

In [4]:
import torch

class SelfAttention_v1(torch.nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        print(f"W_query: {self.W_query}")
        print(f"W_key: {self.W_key}")
        print(f"W_query: {self.W_value}")

    def forward(self, input_matrix):
        queries = self.W_query(input_matrix)
        keys = self.W_key(input_matrix)
        values = self.W_value(input_matrix)

        print(f"Modified Queries: {queries.shape}")
        print(f"Modified keys: {keys.shape}")
        print(f"Modified values: {values.shape}")

        attn_scores = queries @ keys.T
        print(f"attn_scores: {attn_scores.shape}")

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        print(f"attn_weights: {attn_weights.shape}")

        context_vectors = attn_weights @ values
        print(f"context_vectors: {context_vectors.shape}")
        return context_vectors
    
s = SelfAttention_v1(4, 2)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
   [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
   [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
)
s.forward(inputs)

W_query: Linear(in_features=4, out_features=2, bias=False)
W_key: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
Modified Queries: torch.Size([3, 2])
Modified keys: torch.Size([3, 2])
Modified values: torch.Size([3, 2])
attn_scores: torch.Size([3, 3])
attn_weights: torch.Size([3, 3])
context_vectors: torch.Size([3, 2])


tensor([[-0.3490,  0.1014],
        [-0.3489,  0.1011],
        [-0.3490,  0.1010]], grad_fn=<MmBackward0>)

<img src="../Images/causal-attention.png" width=“500”>

In [5]:
import torch

# Without the support of batching
class CausalAttention_v1(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, drop_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(drop_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        print(f"W_query: {self.W_query}")
        print(f"W_key: {self.W_key}")
        print(f"W_query: {self.W_value}")

    def forward(self, input_matrix):
        num_tokens, d_in = input_matrix.shape
        queries = self.W_query(input_matrix)
        keys = self.W_key(input_matrix)
        values = self.W_value(input_matrix)

        print(f"Modified Queries: {queries.shape}")
        print(f"Modified keys: {keys.shape}")
        print(f"Modified values: {values.shape}")

        attn_scores = queries @ keys.T
        print(f"attn_scores: {attn_scores.shape}")

        masked_attn_scores = attn_scores.masked_fill_(
            # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(
            masked_attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        print(f"attn_weights: {attn_weights.shape}")

        context_vectors = attn_weights @ values
        print(f"context_vectors: {context_vectors.shape}")
        return context_vectors
    

inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
   [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
   [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
)
# getting the columns
d_in = inputs.shape[1]

print(f"Total input dimension or context vector = {d_in}")
d_out = 2

print(inputs[0].shape)
print(inputs.shape[1])

# getting the rows
tokens = inputs.shape[0]
s = CausalAttention_v1(d_in, d_out, tokens, 0.5)
s.forward(inputs)

Total input dimension or context vector = 4
torch.Size([4])
4
W_query: Linear(in_features=4, out_features=2, bias=False)
W_key: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
Modified Queries: torch.Size([3, 2])
Modified keys: torch.Size([3, 2])
Modified values: torch.Size([3, 2])
attn_scores: torch.Size([3, 3])
attn_weights: torch.Size([3, 3])
context_vectors: torch.Size([3, 2])


tensor([[-0.0285, -0.2704],
        [-0.2420, -0.3034],
        [-0.2729, -0.2573]], grad_fn=<MmBackward0>)

In [6]:
import torch

# With the support of batching
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, drop_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = torch.nn.Dropout(drop_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        print(f"W_query: {self.W_query}")
        print(f"W_key: {self.W_key}")
        print(f"W_query: {self.W_value}")

    def forward(self, input_matrix):
        batch, num_tokens, d_in = input_matrix.shape
        queries = self.W_query(input_matrix)
        keys = self.W_key(input_matrix)
        values = self.W_value(input_matrix)

        print(f"Modified Queries: {queries.shape}")
        print(f"Modified keys: {keys.shape}")
        print(f"Modified values: {values.shape}")

        print(f"Before transpose: {keys}")
        attn_scores = queries @ keys.transpose(1,2)
        print(f"Simple transpose: {keys.T}")
        print(f"After transpose: {keys.transpose(1,2)}")
        print(f"attn_scores: {attn_scores.shape}")

        masked_attn_scores = attn_scores.masked_fill_(
            # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(
            masked_attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )
        print(f"attn_weights: {attn_weights.shape}")
        attn_weights = self.dropout(attn_weights) # New
        print(f"attn_weights: {attn_weights.shape}")

        context_vectors = attn_weights @ values
        print(f"context_vectors: {context_vectors.shape}")
        return context_vectors
    

inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
   [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
   [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
)
print(f"Inputs: \n\n", inputs, end='\n\n')
inputs = torch.stack((inputs, inputs), dim=0)
print(f"Batch: Stacked Inputs: \n\n", inputs, end='\n\n')

# getting the columns
d_in = inputs[0].shape[1]

print(f"Total input dimension or context vector = {d_in}", end='\n\n')
d_out = 2

print(f"Input shape: {inputs[0].shape}", end='\n\n')
print(f"Input #of rows: {inputs.shape[1]}", end='\n\n')

# getting the rows
tokens = inputs[0].shape[0]
print(f"# of Tokens: {inputs[0].shape[0]}", end='\n\n')
s = CausalAttention(d_in, d_out, tokens, 0.5)
s.forward(inputs)

Inputs: 

 tensor([[0.4300, 0.1500, 0.8900, 0.1200],
        [0.5500, 0.8700, 0.6600, 0.5400],
        [0.5700, 0.8500, 0.6400, 0.1200]])

Batch: Stacked Inputs: 

 tensor([[[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]],

        [[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]]])

Total input dimension or context vector = 4

Input shape: torch.Size([3, 4])

Input #of rows: 3

# of Tokens: 3

W_query: Linear(in_features=4, out_features=2, bias=False)
W_key: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
Modified Queries: torch.Size([2, 3, 2])
Modified keys: torch.Size([2, 3, 2])
Modified values: torch.Size([2, 3, 2])
Before transpose: tensor([[[-0.0255,  0.0891],
         [-0.4465, -0.1261],
         [-0.3684, -0.0899]],

        [[-0.0255,  0.0891],
         [-0.4465, -0.1261],
   

/var/folders/n8/nxj5v7m57_vdk043ccwsymvm0000gn/T/ipykernel_47531/1122040848.py:28: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4483.)
  print(f"Simple transpose: {keys.T}")


tensor([[[ 0.0000,  0.0000],
         [-0.2071,  0.1432],
         [-0.0146,  0.0793]],

        [[ 0.0000,  0.0000],
         [-0.2071,  0.1432],
         [ 0.0000,  0.0000]]], grad_fn=<UnsafeViewBackward0>)

<img src="../Images/multi-head-attention.png" width=“500”>

In [7]:
class MultiHeadAttentionWrapper(torch.nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = torch.nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
   [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
   [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
)
print(f"Inputs: \n\n", inputs, end='\n\n')
inputs = torch.stack((inputs, inputs), dim=0)
print(f"Batch: Stacked Inputs: \n\n", inputs, end='\n\n')

# getting the columns
d_in = inputs[0].shape[1]

print(f"Total input dimension or context vector = {d_in}", end='\n\n')
d_out = 2

print(f"Input shape: {inputs[0].shape}", end='\n\n')
print(f"Input #of rows: {inputs.shape[1]}", end='\n\n')

# getting the rows
tokens = inputs[0].shape[0]
print(f"# of Tokens: {inputs[0].shape[0]}", end='\n\n')

m = MultiHeadAttentionWrapper(d_in=d_in, d_out=d_out, context_length=tokens, dropout=0.5, num_heads=2)
m.forward(inputs)

Inputs: 

 tensor([[0.4300, 0.1500, 0.8900, 0.1200],
        [0.5500, 0.8700, 0.6600, 0.5400],
        [0.5700, 0.8500, 0.6400, 0.1200]])

Batch: Stacked Inputs: 

 tensor([[[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]],

        [[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]]])

Total input dimension or context vector = 4

Input shape: torch.Size([3, 4])

Input #of rows: 3

# of Tokens: 3

W_query: Linear(in_features=4, out_features=2, bias=False)
W_key: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
W_key: Linear(in_features=4, out_features=2, bias=False)
W_query: Linear(in_features=4, out_features=2, bias=False)
Modified Queries: torch.Size([2, 3, 2])
Modified keys: torch.Size([2, 3, 2])
Modified values: torch.Size([2, 

tensor([[[-1.3535,  0.1897,  0.3051,  0.7668],
         [-1.0552,  0.1115,  0.0000,  0.0000],
         [-1.7929,  0.1339,  0.1415, -0.0317]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.1417,  0.3561],
         [-0.5911, -0.0098,  0.1415, -0.0317]]], grad_fn=<CatBackward0>)

In [8]:
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec
    

inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.12], # Your     (x^1)
   [0.55, 0.87, 0.66, 0.54], # journey  (x^2)
   [0.57, 0.85, 0.64, 0.12]] # starts   (x^3)
)
inputs = torch.stack((inputs, inputs), dim=0)
print(f"Batch: Stacked Inputs: \n\n", inputs, end='\n\n')


batch, tokens, d_in = inputs.shape
num_heads = 2

print(f"Batch: {batch}, tokens or context length: {tokens}, input_dimension: {d_in}, num_heads: {num_heads}", end='\n\n')

m = MultiHeadAttention(d_in=d_in, d_out=d_out, context_length=tokens, dropout=0.5, num_heads=num_heads)
m.forward(inputs)

Batch: Stacked Inputs: 

 tensor([[[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]],

        [[0.4300, 0.1500, 0.8900, 0.1200],
         [0.5500, 0.8700, 0.6600, 0.5400],
         [0.5700, 0.8500, 0.6400, 0.1200]]])

Batch: 2, tokens or context length: 3, input_dimension: 4, num_heads: 2



tensor([[[ 0.4286, -0.2890],
         [ 0.3872, -0.3041],
         [ 0.3773, -0.3074]],

        [[ 0.4580, -0.2788],
         [ 0.4961, -0.2651],
         [ 0.3650, -0.3116]]], grad_fn=<ViewBackward0>)

<h2 class="alert alert-block alert-info"> Layer normalization</h2>

In [9]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [10]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

torch.manual_seed(123)
batch_example = torch.randn(2, 5) #A
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


<h2 class="alert alert-block alert-info">GELU activation function</h2>

In [11]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

<h2 class="alert alert-block alert-info">Feed forward network</h2>
<img src="../Images/Feed-Forward-Example.png" width=“500”>

In [12]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768) #A
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


<h2 class="alert alert-block alert-info"> Transformer block </h2>

<img src="../Images/Transformer-block.png" width=“500”>

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x
    
torch.manual_seed(123)
x = torch.rand(2, 4, 768) #A
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


<h2 class="alert alert-block alert-info">GPT Model </h2>

In [14]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device='cpu'))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

batch = []
tokenizer = tiktoken.get_encoding("gpt2")
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)    
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [15]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

out = generate_text_simple(
model=model,
idx=encoded_tensor,
max_new_tokens=6,
context_size=GPT_CONFIG_124M["context_length"]
)
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])
Hello, I am Featureiman Byeswick palpMust
